# 7. Indeks Mie Ayam: Bagian 1 (Kenalan dengan Data)

Notebook 2 sudah menghitung biaya bahan semangkuk dalam rupiah. Di Notebook 7 kita ubah jadi **indeks** (Jan 2021 = 100), supaya mudah dibaca sebagai "naik berapa persen sejak awal" dan bisa dipakai di dashboard.

Bagian 1 ini cuma kenalan dengan data. Belum ada chart dan belum ada kesimpulan.

## Resep asumsi per porsi (sama dengan Notebook 2)
- Ayam 60 g
- Bawang merah 10 g
- Bawang putih 5 g
- Minyak goreng 20 ml

Terigu 100 g tidak masuk indeks utama, mengikuti Notebook 2 sampai 5. Terigu diuji terpisah di Notebook 5b.

## Rumus (rencana)
1. Biaya semangkuk = jumlah dari (harga per satuan x jumlah pakai per porsi)
2. Indeks bulan t = biaya bulan t / biaya Jan 2021 x 100

Satuan harga di data (per kg atau per liter) dicek dulu di bagian 2 sebelum rumus dipakai.

**Cek hasil:** biaya yang kita hitung nanti harus sama dengan Notebook 2 (Rp 2.763 di Jan 2021 dan Rp 3.613 di Sep 2026). Kalau beda, ada yang salah.

In [7]:
import pandas as pd

path = '/kaggle/input/datasets/samuelgeraldo/bedah-mie-ayam-pakai-data-bukan-sumpit/harga_pangan_bersih.csv'
df = pd.read_csv(path)

print(df.shape)
print(df.columns.tolist())
df.head()

(350, 3)
['bulan', 'komoditas', 'harga_rp']


,bulan,komoditas,harga_rp
0,2021-01,Bawang Merah Ukuran Sedang,NaN
1,2021-02,Bawang Merah Ukuran Sedang,31450.0
2,2021-03,Bawang Merah Ukuran Sedang,34250.0
3,2021-04,Bawang Merah Ukuran Sedang,34400.0
4,2021-05,Bawang Merah Ukuran Sedang,32950.0


**Hasilnya:** data punya 350 baris dan 3 kolom: `bulan`, `komoditas`, `harga_rp`. Satu baris = satu harga satu komoditas di satu bulan (format panjang). 350 baris = 5 komoditas x 70 bulan (Jan 2021 sampai Okt 2026).

Lima baris pertama semuanya Bawang Merah Ukuran Sedang. Baris Jan 2021 harganya kosong (`NaN`), sedangkan Feb 2021 sudah Rp 31.450.

In [8]:
df.info()
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   bulan      350 non-null    object 
 1   komoditas  350 non-null    object 
 2   harga_rp   335 non-null    float64
dtypes: float64(1), object(2)
memory usage: 8.3+ KB
bulan         0
komoditas     0
harga_rp     15
dtype: int64


**Hasilnya:**
- `bulan` dan `komoditas` terisi semua, tapi `bulan` masih berupa teks ("2021-01"), bukan tanggal. Diubah di bagian 2.
- `harga_rp` terisi 335 dari 350. Ada **15 harga kosong**.

Dugaan: 15 = 3 bulan x 5 komoditas, yaitu Jan 2021, Mei 2022, dan Jun 2022 yang memang kosong di sumber (PIHPS). Sel berikut mengeceknya.

In [9]:
print(df[df['harga_rp'].isna()].groupby('bulan').size())
print()
print(df['komoditas'].unique())
print()
print(df['bulan'].min(), 'sampai', df['bulan'].max())

bulan
2021-01    5
2022-05    5
2022-06    5
dtype: int64

['Bawang Merah Ukuran Sedang' 'Bawang Putih Ukuran Sedang'
 'Daging Ayam Ras Segar' 'Minyak Goreng Kemasan Bermerk 1'
 'Minyak Goreng Kemasan Bermerk 2']

2021-01 sampai 2026-10


**Hasilnya:**
- **Bulan kosong:** tepat 3 bulan, yaitu Jan 2021, Mei 2022, dan Jun 2022, masing-masing kosong untuk kelima komoditas (3 x 5 = 15). Jadi kosongnya karena sumber memang tidak punya data bulan itu, bukan karena satu komoditas bermasalah.
- **Lima komoditas:** Bawang Merah Ukuran Sedang, Bawang Putih Ukuran Sedang, Daging Ayam Ras Segar, Minyak Goreng Kemasan Bermerk 1, dan Minyak Goreng Kemasan Bermerk 2.
- **Rentang bulan:** 2021-01 sampai 2026-10. Okt 2026 belum utuh, jadi nanti dibuang.

Catatan: resep hanya punya satu bahan minyak goreng (20 ml), tapi di data ada dua merek. Di bagian 2 kita tentukan cara memakainya (misalnya dirata-rata), dan harus sama dengan cara di Notebook 2.

## Rencana bagian 2
- Ubah `bulan` jadi tanggal.
- Isi 3 bulan kosong dan buang Okt 2026, seperti di Notebook 2.
- Tentukan cara memakai dua merek minyak goreng.
- Pastikan satuan harga, lalu hitung biaya semangkuk dan cocokkan dengan Notebook 2 (Rp 2.763 di Jan 2021, Rp 3.613 di Sep 2026).